In [2]:
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------
# Config
# ---------------------------------------
FILE_PATH = "session_dataset.csv"
TARGET = "any_addon_added"
CART_COL = "base_cart_item_count"
OFFER_COL = "has_offer"

# ---------------------------------------
# Load data
# ---------------------------------------
df = pd.read_csv(FILE_PATH)

print("Dataset shape:", df.shape)

# ---------------------------------------
# 1. Conversion by cart size (overall)
# ---------------------------------------
cart_conv = (
    df.groupby(CART_COL)[TARGET]
    .agg(conversion_rate="mean", session_count="count")
    .reset_index()
    .sort_values(CART_COL)
)

print("\n=== Overall conversion by cart size ===")
print(cart_conv)

# ---------------------------------------
# 2. Define small vs large carts
# ---------------------------------------
small_cart = df[df[CART_COL] <= 1]
large_cart = df[df[CART_COL] >= 3]

small_conv = small_cart[TARGET].mean()
large_conv = large_cart[TARGET].mean()

prob_change = large_conv - small_conv

print("\nSmall cart conversion:", small_conv)
print("Large cart conversion:", large_conv)
print("Change in probability:", prob_change)

# ---------------------------------------
# 3. Conditional analysis by offer status
# ---------------------------------------
offer_cart_conv = (
    df.groupby([OFFER_COL, CART_COL])[TARGET]
    .agg(conversion_rate="mean", session_count="count")
    .reset_index()
)

print("\n=== Conversion by cart size AND offer ===")
print(offer_cart_conv)

# ---------------------------------------
# 4. Split analysis
# ---------------------------------------
with_offer = df[df[OFFER_COL] == 1]
without_offer = df[df[OFFER_COL] == 0]

with_offer_conv = (
    with_offer.groupby(CART_COL)[TARGET]
    .mean()
    .reset_index()
)

without_offer_conv = (
    without_offer.groupby(CART_COL)[TARGET]
    .mean()
    .reset_index()
)

print("\n=== Conversion vs cart size (WITH offer) ===")
print(with_offer_conv)

print("\n=== Conversion vs cart size (NO offer) ===")
print(without_offer_conv)

# ---------------------------------------
# 5. Detect Simpson-style reversal
# ---------------------------------------
trend_overall = large_conv > small_conv

trend_with_offer = (
    with_offer[with_offer[CART_COL] >= 3][TARGET].mean() >
    with_offer[with_offer[CART_COL] <= 1][TARGET].mean()
)

trend_without_offer = (
    without_offer[without_offer[CART_COL] >= 3][TARGET].mean() >
    without_offer[without_offer[CART_COL] <= 1][TARGET].mean()
)

simpson_possible = (
    trend_overall and not (trend_with_offer and trend_without_offer)
)

print("\n=== Trend diagnostics ===")
print("Overall trend (large > small):", trend_overall)
print("Trend WITH offers:", trend_with_offer)
print("Trend WITHOUT offers:", trend_without_offer)
print("Possible Simpson's paradox:", simpson_possible)

# ---------------------------------------
# 6. Save tables
# ---------------------------------------
cart_conv.to_csv("q3_conversion_by_cart_size.csv", index=False)
offer_cart_conv.to_csv("q3_conversion_by_cart_and_offer.csv", index=False)

# ---------------------------------------
# 7. Graphs
# ---------------------------------------

# Overall relationship
plt.figure(figsize=(10,6))
plt.plot(cart_conv[CART_COL], cart_conv["conversion_rate"], marker="o")
plt.title("Add-On Conversion Rate vs Cart Size (Overall)")
plt.xlabel("Base Cart Item Count")
plt.ylabel("Conversion Rate")
plt.grid(True)
plt.tight_layout()
plt.savefig("q3_overall_cart_conversion.png", dpi=200)
plt.close()

# Offer vs no-offer comparison
plt.figure(figsize=(10,6))
plt.plot(with_offer_conv[CART_COL], with_offer_conv[TARGET], marker="o", label="Offer")
plt.plot(without_offer_conv[CART_COL], without_offer_conv[TARGET], marker="o", label="No Offer")
plt.title("Add-On Conversion vs Cart Size (Offer vs No Offer)")
plt.xlabel("Base Cart Item Count")
plt.ylabel("Conversion Rate")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("q3_offer_vs_no_offer_cart_conversion.png", dpi=200)
plt.close()

print("\nSaved files:")
print("- q3_conversion_by_cart_size.csv")
print("- q3_conversion_by_cart_and_offer.csv")
print("- q3_overall_cart_conversion.png")
print("- q3_offer_vs_no_offer_cart_conversion.png")

# ---------------------------------------
# 8. Reference summary
# ---------------------------------------
summary = {
    "small_cart_conversion": float(small_conv),
    "large_cart_conversion": float(large_conv),
    "probability_change": float(prob_change),
    "overall_large_cart_better": bool(trend_overall),
    "trend_with_offer": bool(trend_with_offer),
    "trend_without_offer": bool(trend_without_offer),
    "possible_simpsons_paradox": bool(simpson_possible)
}

print("\n=== Reference summary ===")
print(summary)

Dataset shape: (100000, 57)

=== Overall conversion by cart size ===
   base_cart_item_count  conversion_rate  session_count
0                     1         0.205608          40120
1                     2         0.258207          35274
2                     3         0.255555          19757
3                     4         0.248312           4442
4                     5         0.275253            396
5                     6         0.363636             11

Small cart conversion: 0.20560817547357926
Large cart conversion: 0.25461269609038445
Change in probability: 0.049004520616805186

=== Conversion by cart size AND offer ===
    has_offer  base_cart_item_count  conversion_rate  session_count
0           0                     1         0.181619          27877
1           0                     2         0.233533          24823
2           0                     3         0.228846          13756
3           0                     4         0.228133           3064
4           0            